<a href="https://colab.research.google.com/github/khojiakbar05/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
print("Menu Detector")

Menu Detector


In [16]:
# --------------------
# Import Libraries
# --------------------
from google.colab import drive
import torchvision.transforms as transforms
import torch

from torch.utils.data import Dataset
from PIL import Image, UnidentifiedImageError
import os
import numpy as np

In [7]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_path: ', DATASET_PATH)


CUSTOM_CLASS_MAPPING = {    # label grouping | class consolidation
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    #'dessert': 'dessert',
    'cheesecake': 'dessert',
    'cheese_plate': 'dessert',
    'chocolate_cake': 'dessert',
    'pizza': 'pizza'
}


CLASSES = ['hamburger', 'hot_dog', 'dessert', 'cheesecake', 'cheese_plate', 'chocolate_cake', 'pizza']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(NUM_CLASSES)
print(CLASS_TO_IDX)

transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],        # rangler 1-255
                               std=[0.229, 0.224, 0.225])  # rangler 0-1h
])

# 0 ~ 255
# 0.0 ~ 1.0
# H, W, C => C, H, W | RGB = Red, Green, Blue
# Normalize
# pixel = (pixel - mean) / std

Dataset_path:  /content/drive/MyDrive/food101_dataset
7
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'cheesecake': 3, 'cheese_plate': 4, 'chocolate_cake': 5, 'pizza': 6}


Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

In [18]:
# ---------------------------
# Custom Dataset Class
# ---------------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            # Prevent infinite loop if all images are broken
            return self.__getitem__((idx + 1) % len(self.images))

        if self.transform:
            image = self.transform(image)

        return image, label

In [29]:
# -----------------------
# Gather and Split Data
# -----------------------

all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)
    print('class_path: ', class_path)
    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue
    for img in os.listdir(class_path):
        if img.lower().endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

if len(all_images) == 0:
    print("No images found! Please verify your DATASET_PATH and folder structure.")
else:
    np.random.shuffle(all_images)
    split = int(0.8 * len(all_images))
    train_data = all_images[:split]
    val_data = all_images[split:]

    train_images, train_labels = zip(*train_data)
    val_images, val_labels = zip(*val_data)

    print('all_images:', len(all_images))

    dataset = FoodDataset(train_images, train_labels)
    print('len(dataset):', len(dataset))
    img, lbl = dataset[0]
    print('Sample image successfully loaded. Label index:', lbl)

class_path:  /content/drive/MyDrive/food101_dataset/hamburger
class_path:  /content/drive/MyDrive/food101_dataset/hot_dog
class_path:  /content/drive/MyDrive/food101_dataset/cheesecake
class_path:  /content/drive/MyDrive/food101_dataset/cheese_plate
class_path:  /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path:  /content/drive/MyDrive/food101_dataset/pizza
all_images: 6000
Train dataset size: 4800
Sample image successfully loaded. Label index: 2
